In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

BASE_PATH = '/content/drive/Shareddrives/Sieci_Neuronowe/Projekt_1'

with open(f'{BASE_PATH}/porownanie_wynikow.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

with open(f'{BASE_PATH}/long_training_vit_best_150ep.json', 'r', encoding='utf-8') as f:
    long_data = json.load(f)

print('Dane załadowane pomyślnie!')
print(f'Dostępne datasety: {list(data.keys())}')

## 1. Przygotowanie danych

In [ ]:
def extract_results(data_dict):
    results = []
    for item in data_dict:
        num_epochs = len(item.get('history', [])) or item.get('param_value', 1)
        time_per_epoch = item.get('time_per_epoch') or item['train_time_s'] / num_epochs
        results.append({
            'Model': item['model'],
            'Experiment': item['experiment'],
            'Test Accuracy (%)': item['test_acc'],
            'Train Accuracy (%)': item['train_acc'],
            'Training Time (s)': item['train_time_s'],
            'Liczba Parametrów': item['num_params'],
            'Czas/Epoch (s)': time_per_epoch
        })
    return pd.DataFrame(results)

cifar10_df = extract_results(data['cifar10'])
imagenet_df = extract_results(data['imagenet'])

print('CIFAR-10:')
print(cifar10_df[['Model','Experiment','Test Accuracy (%)','Training Time (s)','Liczba Parametrów']])
print('\nImageNet:')
print(imagenet_df[['Model','Experiment','Test Accuracy (%)','Training Time (s)','Liczba Parametrów']])

## 2. Porównanie Test Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [
    (axes[0], cifar10_df, 'CIFAR-10'),
    (axes[1], imagenet_df, 'ImageNet')
]:
    colors = ['#FF6B6B' if m == 'CNN' else '#4ECDC4' for m in df['Model']]
    bars = ax.bar(range(len(df)), df['Test Accuracy (%)'], color=colors, alpha=0.85, edgecolor='black')
    ax.set_title(f'{title}: Test Accuracy', fontsize=12, fontweight='bold')
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['Experiment'], rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(df['Test Accuracy (%)']):
        ax.text(i, v + 0.5, f'{v:.2f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
    # Legenda kolorów
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#FF6B6B', label='CNN'), Patch(color='#4ECDC4', label='ViT')], fontsize=10)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/01_test_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Porównanie Czasu Treningu

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [
    (axes[0], cifar10_df, 'CIFAR-10'),
    (axes[1], imagenet_df, 'ImageNet')
]:
    colors = ['#FF6B6B' if m == 'CNN' else '#4ECDC4' for m in df['Model']]
    ax.bar(range(len(df)), df['Training Time (s)'], color=colors, alpha=0.85, edgecolor='black')
    ax.set_title(f'{title}: Czas Treningu', fontsize=12, fontweight='bold')
    ax.set_ylabel('Czas Treningu (s)', fontsize=11)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['Experiment'], rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(df['Training Time (s)']):
        ax.text(i, v + max(df['Training Time (s)'])*0.02, f'{v:.0f}s', ha='center', va='bottom', fontsize=8, fontweight='bold')
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#FF6B6B', label='CNN'), Patch(color='#4ECDC4', label='ViT')], fontsize=10)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/02_training_time.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Porównanie Liczby Parametrów

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Użyj CIFAR-10 (parametry są takie same dla ImageNet)
colors = ['#FF6B6B' if m == 'CNN' else '#4ECDC4' for m in cifar10_df['Model']]
ax.bar(range(len(cifar10_df)), cifar10_df['Liczba Parametrów'] / 1e6, color=colors, alpha=0.85, edgecolor='black')
ax.set_title('Liczba Parametrów Modeli', fontsize=12, fontweight='bold')
ax.set_ylabel('Parametry (miliony)', fontsize=11)
ax.set_xticks(range(len(cifar10_df)))
ax.set_xticklabels(cifar10_df['Experiment'], rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(cifar10_df['Liczba Parametrów'] / 1e6):
    ax.text(i, v + 0.1, f'{v:.2f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#FF6B6B', label='CNN'), Patch(color='#4ECDC4', label='ViT')], fontsize=10)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/03_parameters.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Krzywe Treningu — CIFAR-10 (Accuracy | Loss osobno)

In [ ]:
items = data['cifar10']
n = len(items)
fig, axes = plt.subplots(2, n, figsize=(n * 4, 8))
fig.suptitle('CIFAR-10: Krzywe Treningu  (górny rząd = Accuracy | dolny rząd = Loss)',
             fontsize=13, fontweight='bold')

for idx, item in enumerate(items):
    h = item['history']
    epochs    = [x['epoch']     for x in h]
    train_acc = [x['train_acc'] for x in h]
    test_acc  = [x['test_acc']  for x in h]
    loss      = [x['loss']      for x in h]
    color = '#FF6B6B' if item['model'] == 'CNN' else '#4ECDC4'
    label = f"{item['model']}\n{item['experiment']}"

    # --- Accuracy ---
    axes[0, idx].plot(epochs, train_acc, 'o-', label='Train Acc', color='#2E86AB', lw=2, markersize=5)
    axes[0, idx].plot(epochs, test_acc,  's-', label='Test Acc',  color='#A23B72', lw=2, markersize=5)
    axes[0, idx].set_title(label, fontsize=10, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))
    axes[0, idx].set_ylabel('Accuracy (%)' if idx == 0 else '')
    axes[0, idx].set_ylim(0, 105)
    axes[0, idx].legend(fontsize=8, loc='lower right')
    axes[0, idx].grid(alpha=0.3)

    # --- Loss ---
    axes[1, idx].plot(epochs, loss, '^-', color='#F18F01', lw=2, markersize=5, label='Loss')
    axes[1, idx].set_title(label, fontsize=10, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('Loss' if idx == 0 else '')
    axes[1, idx].legend(fontsize=8, loc='upper right')
    axes[1, idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/04_cifar10_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Krzywe Treningu — ImageNet (Accuracy | Loss osobno)

In [ ]:
items = data['imagenet']
n = len(items)
fig, axes = plt.subplots(2, n, figsize=(n * 4, 8))
fig.suptitle('ImageNet: Krzywe Treningu  (górny rząd = Accuracy | dolny rząd = Loss)',
             fontsize=13, fontweight='bold')

for idx, item in enumerate(items):
    h = item['history']
    epochs    = [x['epoch']     for x in h]
    train_acc = [x['train_acc'] for x in h]
    test_acc  = [x['test_acc']  for x in h]
    loss      = [x['loss']      for x in h]
    color = '#FF6B6B' if item['model'] == 'CNN' else '#4ECDC4'
    label = f"{item['model']}\n{item['experiment']}"

    # --- Accuracy ---
    axes[0, idx].plot(epochs, train_acc, 'o-', label='Train Acc', color='#2E86AB', lw=2, markersize=5)
    axes[0, idx].plot(epochs, test_acc,  's-', label='Test Acc',  color='#A23B72', lw=2, markersize=5)
    axes[0, idx].set_title(label, fontsize=10, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))
    axes[0, idx].set_ylabel('Accuracy (%)' if idx == 0 else '')
    axes[0, idx].set_ylim(0, 105)
    axes[0, idx].legend(fontsize=8, loc='lower right')
    axes[0, idx].grid(alpha=0.3)

    # --- Loss ---
    axes[1, idx].plot(epochs, loss, '^-', color='#F18F01', lw=2, markersize=5, label='Loss')
    axes[1, idx].set_title(label, fontsize=10, fontweight='bold',
                           bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('Loss' if idx == 0 else '')
    axes[1, idx].legend(fontsize=8, loc='upper right')
    axes[1, idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/05_imagenet_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. BONUS: ViT_Best — 150 Epok na CIFAR-10

In [ ]:
long = long_data['cifar10']
h = long['history']
epochs    = [x['epoch']     for x in h]
train_acc = [x['train_acc'] for x in h]
test_acc  = [x['test_acc']  for x in h]
loss      = [x['loss']      for x in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'ViT_Best: 150 Epochs na CIFAR-10  (best: {long["best_test_acc"]:.2f}% @ ep {long["best_epoch"]})',
             fontsize=13, fontweight='bold')

# Accuracy
axes[0].plot(epochs, train_acc, 'o-', label='Train Acc', color='#2E86AB', lw=2, markersize=4)
axes[0].plot(epochs, test_acc,  's-', label='Test Acc',  color='#A23B72', lw=2, markersize=4)
axes[0].scatter([long['best_epoch']], [long['best_test_acc']],
                s=200, color='gold', edgecolor='black', zorder=5, label=f'Best: {long["best_test_acc"]:.2f}%')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs, loss, '^-', color='#F18F01', lw=2, markersize=4, label='Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE_PATH}/06_vit_150epochs.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Params: {long["num_params"]/1e6:.2f}M  |  Czas: {long["train_time_s"]/3600:.1f}h  |  Best acc: {long["best_test_acc"]:.2f}%')

## 8. Podsumowanie

In [ ]:
print('='*70)
print('PODSUMOWANIE WYNIKÓW')
print('='*70)

for dataset, df in [('CIFAR-10', cifar10_df), ('ImageNet', imagenet_df)]:
    best_cnn = df[df['Model'] == 'CNN']['Test Accuracy (%)'].max()
    best_vit = df[df['Model'] == 'ViT']['Test Accuracy (%)'].max()
    winner = 'CNN' if best_cnn > best_vit else 'ViT'
    print(f'\n{dataset}:')
    print(f'  Best CNN: {best_cnn:.2f}%')
    print(f'  Best ViT: {best_vit:.2f}%')
    print(f'  Zwycięzca: {winner}  (różnica: {abs(best_cnn - best_vit):.2f}%)')

print(f'\nBONUS - ViT_Best 150 epok (CIFAR-10):')
print(f'  Best Test Acc: {long_data["cifar10"]["best_test_acc"]:.2f}% @ epoch {long_data["cifar10"]["best_epoch"]}')
print(f'  Czas treningu: {long_data["cifar10"]["train_time_s"]/3600:.1f}h')
print('='*70)